# OpenPlaque — User-Friendly Plaque + PCAT Visualization v3

This version keeps the validated quantitative endpoints unchanged and improves only the **presentation layer**.

**Changes from v2**
- Plaque is filtered at the **connected-component level**, not voxel-by-voxel. A plaque component is displayed in full when it has meaningful adjacency to the predicted coronary vessel. This is intended to remove remote false-positive islands without deleting most of a real plaque component.
- Hotspots are ranked as **unique 3-D plaque components**, so one lesion cannot appear multiple times.
- Empty regions are never promoted as hotspots.
- The report explicitly records how much canonical plaque is retained for display; a warning is shown if a vessel retains <70%.
- RCA PCAT cross-sections, longitudinal ribbon, and corrected geometry-range dashboard are retained from v2.

The component filter is **visualization-only**. It does not alter canonical TPV.

PCAT attenuation is an imaging surrogate related to perivascular inflammation; it is not a direct inflammation measurement and is not Caristo FAI-Score. Research use only.


In [ ]:
# FIRST EXECUTABLE CELL — mount Google Drive first.
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!rm -rf /content/OpenPlaque
!git clone -q --branch user-friendly-visualization-v3-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
!pip -q install -r /content/OpenPlaque/requirements-colab.txt
print('Repository and requirements ready.')


In [ ]:
import os, sys, shutil, zipfile, base64
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, SimpleITK as sitk
from scipy import ndimage as ndi
from scipy.ndimage import map_coordinates
from IPython.display import display, HTML

REPO=Path('/content/OpenPlaque'); sys.path.insert(0,str(REPO/'src'))
ROOT=Path('/content/drive/MyDrive/OpenPlaque')
OUT=ROOT/'User_Friendly_Plaque_PCAT_Report_v3'; OUT.mkdir(parents=True,exist_ok=True)

os.environ['nnUNet_raw']='/content/nnUNet_raw'
os.environ['nnUNet_preprocessed']='/content/nnUNet_preprocessed'
os.environ['nnUNet_results']='/content/nnUNet_results'
for d in [os.environ['nnUNet_raw'],os.environ['nnUNet_preprocessed'],os.environ['nnUNet_results']]:
    Path(d).mkdir(parents=True,exist_ok=True)

model_zip=ROOT/'models'/'Dataset001_CCTA_DHM-20260703T233210Z-3-001.zip'
model_target=Path('/content/nnUNet_results/Dataset001_CCTA_DHM')
if not model_target.exists():
    if not model_zip.exists(): raise FileNotFoundError(model_zip)
    with zipfile.ZipFile(model_zip) as z: z.extractall('/content/nnUNet_results')

drive_zip=ROOT/'Full_DICOM.zip'; local_zip=Path('/content/Full_DICOM.zip')
if not drive_zip.exists(): raise FileNotFoundError(drive_zip)
if not local_zip.exists() or local_zip.stat().st_size!=drive_zip.stat().st_size:
    shutil.copyfile(drive_zip,local_zip)

from openplaque.study import OpenPlaqueStudy
shutil.rmtree('/content/full_dicom_friendly_v3',ignore_errors=True)
study=OpenPlaqueStudy(str(local_zip),extract_root='/content/full_dicom_friendly_v3')
print('Output:',OUT)


## 1. Canonical plaque segmentation + component-level display filter

A canonical refined plaque component is retained **whole** for display if it has enough voxels close to the predicted vessel.

Current display rule:
- compute 3-D connected plaque components;
- define "near vessel" as within 3.0 mm of the nnU-Net vessel label;
- retain a component if it has at least **3 near-vessel voxels** or at least **2% of its voxels** within that neighborhood.

This rule affects visualization only.


In [ ]:
from openplaque.segmentation import segment_vessel
from openplaque.boundary import refine_plaque_mask
from openplaque.artery_detection import detect_artery_series

fallback={'RCA':1035,'LCX':1039,'LAD':1043}
series_map,_=detect_artery_series(study,fallback_series=fallback,return_candidates=True)
print('Detected series:',series_map)

reports=[]
for vessel in ['LAD','RCA','LCX']:
    image,volume,_=study.load_series(series_map[vessel])
    print('Segmenting',vessel,'series',series_map[vessel])
    reports.append(segment_vessel(image,volume,vessel))

def canonical_refine(r):
    return refine_plaque_mask(
        volume=r.volume, mask=r.mask, spacing=r.mask_image.GetSpacing(),
        remove_small=True, min_component_voxels=10,
        trim_lumen_adjacent=True, lumen_distance_voxels=1,
        erode_core=False, high_hu_threshold=None, low_hu_threshold=None
    )

canonical={r.name:canonical_refine(r) for r in reports}

DISPLAY_NEAR_MM=3.0
MIN_NEAR_VOXELS=3
MIN_NEAR_FRACTION=0.02
structure=np.ones((3,3,3),dtype=bool)

display_masks={}
component_tables={}
qc_rows=[]

for r in reports:
    plaque=(canonical[r.name].refined_mask==2)
    vessel=(r.mask==1)
    sp_zyx=np.array(r.mask_image.GetSpacing(),float)[::-1]
    dist=ndi.distance_transform_edt(~vessel,sampling=sp_zyx)
    near=(dist<=DISPLAY_NEAR_MM)

    labels,ncomp=ndi.label(plaque,structure=structure)
    keep=np.zeros_like(plaque,dtype=bool)
    rows=[]
    for cid in range(1,ncomp+1):
        comp=(labels==cid)
        n=int(comp.sum())
        if n==0: continue
        nnear=int((comp & near).sum())
        frac=nnear/n
        retain=(nnear>=MIN_NEAR_VOXELS) or (frac>=MIN_NEAR_FRACTION)
        if retain: keep |= comp

        z_counts=np.sum(comp,axis=(1,2))
        z_peak=int(np.argmax(z_counts))
        coords=np.argwhere(comp)
        centroid=coords.mean(axis=0)
        rows.append({
            'vessel':r.name,'component_id':cid,'component_voxels':n,
            'near_vessel_voxels':nnear,'near_vessel_fraction':frac,
            'retained_for_display':bool(retain),'peak_frame':z_peak,
            'peak_frame_voxels':int(z_counts[z_peak]),
            'centroid_z':float(centroid[0]),'centroid_y':float(centroid[1]),'centroid_x':float(centroid[2])
        })

    tab=pd.DataFrame(rows)
    component_tables[r.name]=tab
    display_masks[r.name]=keep
    retained=int(keep.sum()); total=int(plaque.sum())
    qc_rows.append({
        'vessel':r.name,
        'canonical_plaque_voxels':total,
        'display_retained_voxels':retained,
        'display_retained_pct':100*retained/max(1,total),
        'components_total':int(ncomp),
        'components_retained':int(tab.retained_for_display.sum()) if len(tab) else 0,
        'display_warning':bool(100*retained/max(1,total)<70)
    })

display_stats=pd.DataFrame(qc_rows)
display_stats.to_csv(OUT/'display_filter_qc_v3.csv',index=False)
pd.concat(component_tables.values(),ignore_index=True).to_csv(OUT/'plaque_component_qc_v3.csv',index=False)
display(display_stats)
if display_stats.display_warning.any():
    print('WARNING: one or more vessels retain <70% of canonical plaque for display.')
else:
    print('PASS: all vessels retain >=70% of canonical plaque for display.')


## 2. Unique component-based plaque hotspot gallery

The top three **retained 3-D plaque components** in each vessel are shown. Each component appears at most once. Empty fallback panels are labeled rather than pretending to be lesions.


In [ ]:
hotspot_rows=[]
fig,axs=plt.subplots(3,3,figsize=(14,14))

for row,r in enumerate(reports):
    tab=component_tables[r.name]
    tab=tab[tab.retained_for_display].sort_values('component_voxels',ascending=False).head(3).copy()
    voxel_mm3=float(np.prod(r.mask_image.GetSpacing()))
    plaque=(canonical[r.name].refined_mask==2)

    for col in range(3):
        ax=axs[row,col]
        if col>=len(tab):
            ax.axis('off')
            ax.text(.5,.5,f'{r.name}\nNo additional retained component',ha='center',va='center',transform=ax.transAxes)
            continue

        rec=tab.iloc[col]
        cid=int(rec.component_id)
        # Re-label to reconstruct this exact component.
        labels,_=ndi.label(plaque,structure=structure)
        comp=(labels==cid)
        z=int(rec.peak_frame)
        coords=np.argwhere(comp[z])
        if len(coords):
            cy,cx=coords.mean(axis=0)
        else:
            cy,cx=r.volume.shape[1]/2,r.volume.shape[2]/2

        half=70
        y0=max(0,int(round(cy))-half); y1=min(r.volume.shape[1],int(round(cy))+half)
        x0=max(0,int(round(cx))-half); x1=min(r.volume.shape[2],int(round(cx))+half)

        img=r.volume[z,y0:y1,x0:x1]
        cm=comp[z,y0:y1,x0:x1]
        vm=(r.mask[z,y0:y1,x0:x1]==1)

        ax.imshow(img,cmap='gray',vmin=-200,vmax=800)
        ax.imshow(np.ma.masked_where(~cm,cm),alpha=.58,cmap='autumn',interpolation='nearest')
        if np.any(cm): ax.contour(cm,levels=[0.5],linewidths=1.25)
        if np.any(vm): ax.contour(vm,levels=[0.5],linewidths=.8,linestyles='--')
        vol=float(rec.component_voxels*voxel_mm3)
        ax.set_title(f'{r.name} component {col+1}\n{vol:.1f} mm³; frame {z}')
        ax.axis('off')

        hotspot_rows.append({
            'vessel':r.name,'rank':col+1,'component_id':cid,'peak_frame':z,
            'component_voxels':int(rec.component_voxels),
            'component_volume_mm3':vol,
            'near_vessel_voxels':int(rec.near_vessel_voxels),
            'near_vessel_fraction':float(rec.near_vessel_fraction)
        })

fig.suptitle('Top unique coronary-adjacent plaque components\norange=plaque, dashed=vessel label; display filter only',fontsize=15)
plt.tight_layout(rect=[0,0,1,.95])
plt.savefig(OUT/'01_plaque_component_hotspot_gallery_v3.png',dpi=180,bbox_inches='tight')
plt.show(); plt.close(fig)

hotspots=pd.DataFrame(hotspot_rows)
hotspots.to_csv(OUT/'top_plaque_components_v3.csv',index=False)
display(hotspots)


## 3. Artery-centered RCA PCAT cross-sections

These are source-CCTA planes perpendicular to the frozen RCA centerline. Solid circle = lumen radius, dashed = modeled outer wall, dotted = canonical shell edge.


In [ ]:
BASE=ROOT/'PCAT_RCA_10_50'
cp=BASE/'rca_centerline_smoothed_zyx.csv'
rp=BASE/'pcat_local_radius_profile.csv'
if not cp.exists() or not rp.exists():
    raise FileNotFoundError('Missing frozen RCA PCAT centerline/radius inputs.')

source_img,ct,_=study.load_series(7)
ct=np.asarray(ct,float)
sp_xyz=np.array(source_img.GetSpacing(),float)
sp_zyx=sp_xyz[::-1]

cl=pd.read_csv(cp); rad=pd.read_csv(rp)
arc=cl.arc_mm.to_numpy(float)
pts_zyx=cl[['z','y','x']].to_numpy(float)
pts_mm=pts_zyx*sp_zyx
lumen=np.interp(arc,rad.arc_mm.to_numpy(float),rad.lumen_radius_mm.to_numpy(float))

aorta_candidates=[
    ROOT/'RCA_Ostium_TotalSegmentator'/'aorta_series7_totalseg.nii.gz',
    ROOT/'TotalSegmentator_Validation_v2'/'aorta_series7_totalseg.nii.gz'
]
ap=next((p for p in aorta_candidates if p.exists()),None)
if ap is None: raise FileNotFoundError('Missing TotalSegmentator aorta mask.')
ai=sitk.ReadImage(str(ap))
if ai.GetSize()!=source_img.GetSize() or not np.allclose(ai.GetSpacing(),source_img.GetSpacing()):
    ai=sitk.Resample(ai,source_img,sitk.Transform(),sitk.sitkNearestNeighbor,0,sitk.sitkUInt8)
aorta=sitk.GetArrayFromImage(ai).astype(float)

def plane_basis(t):
    t=np.asarray(t,float); t=t/np.linalg.norm(t)
    ref=np.array([1.,0.,0.]) if abs(t[0])<.85 else np.array([0.,1.,0.])
    u=np.cross(t,ref); u=u/np.linalg.norm(u)
    v=np.cross(t,u); return u,v/np.linalg.norm(v)

def sample_plane(target,half=8.,pix=.20):
    i=int(np.argmin(np.abs(arc-target)))
    i0=max(0,i-2); i1=min(len(arc)-1,i+2)
    tangent=pts_mm[i1]-pts_mm[i0]; tangent=tangent/np.linalg.norm(tangent)
    u,v=plane_basis(tangent); center=pts_mm[i]
    c=np.arange(-half,half+1e-9,pix)
    U,V=np.meshgrid(c,c,indexing='xy')
    xyz=center[None,None,:]+U[...,None]*u+V[...,None]*v
    vox=(xyz/sp_zyx).reshape(-1,3).T
    img=map_coordinates(ct,vox,order=1,mode='nearest').reshape(U.shape)
    am=map_coordinates(aorta,vox,order=0,mode='nearest').reshape(U.shape)>0.5
    rr=np.sqrt(U**2+V**2)
    lum=float(lumen[i]); outer=lum+.75; shell_outer=3*outer
    fat=(rr>outer)&(rr<=shell_outer)&(~am)&(img>=-190)&(img<=-30)
    return {'arc':float(arc[i]),'img':img,'fat':fat,'lumen':lum,'outer':outer,'shell_outer':shell_outer,'extent':[-half,half,-half,half]}

planes=[sample_plane(x) for x in [10,20,30,40,50]]
fig,axs=plt.subplots(1,5,figsize=(20,4.5))
for ax,d,target in zip(axs,planes,[10,20,30,40,50]):
    ax.imshow(d['img'],cmap='gray',vmin=-200,vmax=800,extent=d['extent'],origin='lower')
    fi=np.ma.masked_where(~d['fat'],d['img'])
    im=ax.imshow(fi,cmap='coolwarm',vmin=-120,vmax=-60,alpha=.8,extent=d['extent'],origin='lower')
    for rr,ls in [(d['lumen'],'-'),(d['outer'],'--'),(d['shell_outer'],':')]:
        ax.add_patch(plt.Circle((0,0),rr,fill=False,linestyle=ls,linewidth=1.7))
    ax.plot(0,0,'+',markersize=9)
    ax.set_title(f'RCA {target} mm\nactual {d["arc"]:.1f} mm')
    ax.set_xlim(-8,8); ax.set_ylim(-8,8); ax.set_aspect('equal'); ax.set_xlabel('mm')
axs[0].set_ylabel('mm')
fig.suptitle('RCA artery-centered PCAT: solid=lumen, dashed=modeled outer wall, dotted=shell edge',fontsize=13)
cb=fig.colorbar(im,ax=axs.ravel().tolist(),shrink=.78,pad=.02); cb.set_label('PCAT attenuation (HU)')
plt.savefig(OUT/'02_rca_pcat_cross_sections_v3.png',dpi=180,bbox_inches='tight')
plt.show(); plt.close(fig)


## 4. Longitudinal PCAT ribbon + summary dashboard


In [ ]:
COMB=ROOT/'Combined_TPV_PCAT_All_Metrics_v2'
lp=COMB/'pcat_canonical_longitudinal_v2.csv'
if not lp.exists():
    raise FileNotFoundError(lp)

longdf=pd.read_csv(lp)
x=(longdf.arc_start_mm.to_numpy(float)+longdf.arc_end_mm.to_numpy(float))/2
y=longdf.mean_hu.to_numpy(float)

fig,ax=plt.subplots(figsize=(12,3.2))
sc=ax.scatter(x,np.zeros_like(x),c=y,cmap='coolwarm',vmin=-110,vmax=-75,s=260,marker='s')
ax.set_xlim(10,50); ax.set_yticks([]); ax.set_xlabel('Distance from RCA ostium (mm)')
ax.set_title('RCA longitudinal PCAT attenuation ribbon')
cb=fig.colorbar(sc,ax=ax,pad=.02); cb.set_label('Mean HU per 1-mm segment')
plt.tight_layout(); plt.savefig(OUT/'03_rca_longitudinal_pcat_ribbon_v3.png',dpi=180,bbox_inches='tight')
plt.show(); plt.close(fig)

tpv=pd.read_csv(COMB/'tpv_metrics_by_vessel_v2.csv')
pcat=pd.read_csv(COMB/'pcat_canonical_primary_v2.csv')
ps=pd.read_csv(COMB/'pcat_circular_sensitivity_v2.csv')
total=tpv[tpv.vessel=='TOTAL'].iloc[0]
primary=pcat.iloc[0]
cmin=float(ps.pcat_mean_hu.min()); cmax=float(ps.pcat_mean_hu.max())

directional=np.nan
dp=ROOT/'PCAT_RCA_10_50_Directional_OuterWall'/'directional_pcat_summary.csv'
if dp.exists():
    d=pd.read_csv(dp)
    if 'directional_pcat_mean_hu' in d.columns:
        directional=float(d.iloc[0]['directional_pcat_mean_hu'])
fmin=min(cmin,directional) if np.isfinite(directional) else cmin
fmax=max(cmax,directional) if np.isfinite(directional) else cmax

fig=plt.figure(figsize=(13,8))
gs=fig.add_gridspec(2,2,height_ratios=[1,1.1])
ax1=fig.add_subplot(gs[0,0])
vv=tpv[tpv.vessel!='TOTAL']
ax1.bar(vv.vessel,vv.canonical_refined_tpv_mm3)
ax1.set_ylabel('Refined TPV (mm³)'); ax1.set_title('Plaque volume by artery')

ax2=fig.add_subplot(gs[0,1]); ax2.axis('off')
retain_txt='\n'.join([f"{r.vessel}: {r.display_retained_pct:.0f}% display-retained" for _,r in display_stats.iterrows()])
txt=(
    f"Total refined TPV: {total.canonical_refined_tpv_mm3:.0f} mm³\n"
    f"Raw TPV: {total.raw_tpv_mm3:.0f} mm³\n"
    f"TPV sensitivity: {total.sensitivity_min_mm3:.0f}–{total.sensitivity_max_mm3:.0f} mm³\n\n"
    f"RCA 10–50 mm PCAT: {primary.pcat_mean_hu:.2f} HU\n"
    f"Circular-margin range: {cmin:.2f} to {cmax:.2f} HU\n"
    f"Full tested geometry: {fmin:.2f} to {fmax:.2f} HU\n\n"
    f"Display filter QC:\n{retain_txt}"
)
ax2.text(.02,.98,txt,va='top',fontsize=13)

ax3=fig.add_subplot(gs[1,:])
ax3.plot(x,y,marker='o',markersize=3)
ax3.axhline(float(primary.pcat_mean_hu),linestyle='--',linewidth=1)
ax3.set_xlim(10,50); ax3.set_xlabel('Distance from RCA ostium (mm)')
ax3.set_ylabel('Mean PCAT HU'); ax3.set_title('RCA longitudinal PCAT profile')
plt.tight_layout()
plt.savefig(OUT/'04_summary_dashboard_v3.png',dpi=180,bbox_inches='tight')
plt.show(); plt.close(fig)


## 5. Export report and compact report-back ZIP


In [ ]:
summary=pd.DataFrame([{
    'total_refined_tpv_mm3':float(total.canonical_refined_tpv_mm3),
    'total_raw_tpv_mm3':float(total.raw_tpv_mm3),
    'tpv_sensitivity_min_mm3':float(total.sensitivity_min_mm3),
    'tpv_sensitivity_max_mm3':float(total.sensitivity_max_mm3),
    'rca_pcat_mean_hu':float(primary.pcat_mean_hu),
    'circular_geometry_min_hu':cmin,
    'circular_geometry_max_hu':cmax,
    'full_geometry_min_hu':fmin,
    'full_geometry_max_hu':fmax,
    'lad_display_retained_pct':float(display_stats.loc[display_stats.vessel=='LAD','display_retained_pct'].iloc[0]),
    'rca_display_retained_pct':float(display_stats.loc[display_stats.vessel=='RCA','display_retained_pct'].iloc[0]),
    'lcx_display_retained_pct':float(display_stats.loc[display_stats.vessel=='LCX','display_retained_pct'].iloc[0])
}])
summary.to_csv(OUT/'friendly_report_summary_v3.csv',index=False)

def img_b64(path):
    return base64.b64encode(Path(path).read_bytes()).decode('ascii')

imgs=[
    ('Plaque component hotspots','01_plaque_component_hotspot_gallery_v3.png'),
    ('RCA artery-centered PCAT','02_rca_pcat_cross_sections_v3.png'),
    ('RCA longitudinal PCAT','03_rca_longitudinal_pcat_ribbon_v3.png'),
    ('Summary dashboard','04_summary_dashboard_v3.png')
]
html=["<html><head><meta charset='utf-8'><title>OpenPlaque Friendly Visual Report v3</title>",
      "<style>body{font-family:Arial;max-width:1200px;margin:30px auto;color:#222}img{width:100%;margin:10px 0 30px}table{border-collapse:collapse}td,th{border:1px solid #bbb;padding:6px} .warn{background:#fff3cd;padding:10px}</style></head><body>",
      "<h1>OpenPlaque Friendly Visual Report v3</h1>",
      "<p><b>Research use only.</b> PCAT attenuation is an imaging surrogate related to perivascular inflammation, not a direct inflammation measurement.</p>",
      "<h2>Display filter QC</h2>",display_stats.to_html(index=False,float_format=lambda x:f'{x:.1f}'),
      "<p>The component filter is visualization-only and does not change canonical TPV.</p>",
      "<h2>Top plaque components</h2>",hotspots.to_html(index=False,float_format=lambda x:f'{x:.2f}')]
for title,name in imgs:
    html += [f"<h2>{title}</h2>",f"<img src='data:image/png;base64,{img_b64(OUT/name)}'>"]
html += ["</body></html>"]
html_path=OUT/'OPENPLAQUE_FRIENDLY_VISUAL_REPORT_V3.html'
html_path.write_text(''.join(html),encoding='utf-8')

zip_path=OUT/'OPENPLAQUE_FRIENDLY_VISUAL_V3_REPORT_BACK.zip'
to_zip=[
    '01_plaque_component_hotspot_gallery_v3.png',
    '02_rca_pcat_cross_sections_v3.png',
    '03_rca_longitudinal_pcat_ribbon_v3.png',
    '04_summary_dashboard_v3.png',
    'display_filter_qc_v3.csv',
    'plaque_component_qc_v3.csv',
    'top_plaque_components_v3.csv',
    'friendly_report_summary_v3.csv',
    'OPENPLAQUE_FRIENDLY_VISUAL_REPORT_V3.html'
]
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for name in to_zip:
        z.write(OUT/name,arcname=name)

print('Report:',html_path)
print('ZIP:',zip_path)
print('Done.')
